# Capstone EDA — Supervised-Learning-Readiness Notebook
## GA 5800 — Summer 2026

**Student:** *Mustafaiz*  
**Dataset:** *The HIG Dataset*  
**Date submitted:** *YYYY-MM-DD*

---

This notebook is your scaffold for the 10-stage capstone. Each stage has:
- a markdown cell describing what to produce,
- empty code cells for your analysis,
- a markdown cell for your written interpretation.

You are **not** limited to the cells provided. Add code, markdown, and plots as needed.

### Before you begin
- Read the full assignment brief: `capstone_eda_supervised_readiness.pdf`.
- Set your random seed in the setup cell below.
- Treat the Stage 10 Data Card as a *living* document — fill it in as you complete each stage, not at the end.

### Submission reminders
- Every p-value must be paired with an **effect size**.
- Every plot must have a title, axis labels (with units), and a caption stating the takeaway.
- Notebook must execute top-to-bottom on a fresh kernel.
- Submit alongside `<lastname>_<firstname>_data_card.md` and `<lastname>_<firstname>_readiness_memo.pdf` as a single zipped folder.

---

## Setup

Imports, random seed, and data load.

In [7]:
# Core
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Stats
from scipy import stats
from scipy.stats import (
    chi2_contingency, kruskal, mannwhitneyu, f_oneway,
    pointbiserialr, spearmanr, kendalltau, pearsonr,
)

# sklearn
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, IsolationForest
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

# Multicollinearity (VIF)
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Display + plot defaultsy
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (8, 5)

In [11]:
# Load the dataset
DATA_PATH = 'UConn_2026_data.csv'  # update this path
df = pd.read_csv(DATA_PATH)

print(f'Shape: {df.shape}')
df.head()

Shape: (300000, 44)


,AUTO_POL_TRANS_ID,RISK_ST_ABBR,POL_EFF_DT,POL_EXP_DT,POL_NEW_RENEW_CD,POL_TTL_BILL_PREM_AMT,RISK_GRP,RISK_FLAG,CLTV_LONGVTY_GRP_CD,AARP_MEMB_YR_CNT,AARP_MEMB_IND,FULL_PAY_DISC_IND,POL_BI_OCCUR_LMT_CD,CLEAN_DIRTY_DESC,DRVR_TLMTC_PGM_TYP_CD,RENEW_YRS,POL_MULTI_SNGL_VEH_CD,POL_DRVR_TLMTC_ENROL_IND,ORIG_INTRNT_RATE_QTE_IND,PPV_CNT,PCARR_DESC,AVG_VEH_PREM,PCARR_YR_CNT,PCARR_CALC_YR_CNT,POL_FORM_CD,POL_RATE_DRVR_MIN_AGE,POL_RATE_DRVR_MAX_AGE,HH_COMP,ADV_QTE_DAY_CNT,AQD_GRP,YOUTHFUL_IND,BILL_PYMNT_METH_DESC,BILL_PYMENT_FREQ_DESC,AVG_VEHICLE_MILEAGE,MKT_CHNNL_NM,MKTG_MEDIA_GRP_DESC,MKTG_NON_BRAND_SEARCH_FLAG,ORIG_QTE_CNTCT_METH_CD,E_SIGN_IND,ECNSNT_BEGINING_OF_POL_TERM_IND,PLCY_MIN_CAR_BASE_PRICE,PLCY_MAX_CAR_BASE_PRICE,PLCY_AVG_CAR_BASE_PRICE,POL_ACCT_CR_IND
0,1064231476,CO,5/12/2025,11/12/2025,R,864.0,6,NaN,PL16,1,Y,1.0,500.0,Dirty,M,1.0,S,N,N,1,Others,864.00,2.0,2.9,0,63,63,SC1D,50.0,14-60,N,CreditOrDebitCard,PayInFull,10000,Search,AARP Branded,Other than Non-Branded Search,P,0,1,23245.0,23245.0,23245.0,0
1,1018298329,WI,3/2/2023,9/2/2023,N,287.0,9,NaN,PL22,1,Y,1.0,300.0,Dirty,X,0.0,S,N,Y,1,Others,287.00,1.0,0.9,H*,24,59,SCMT1D,25.0,14-60,Y,CreditOrDebitCard,RepetitiveFullPay,8000,Online,HIG.com,Other than Non-Branded Search,I,1,1,27170.0,27170.0,27170.0,0
2,1057068242,FL,9/24/2024,3/24/2025,R,980.0,5,NaN,NaN,8,Y,0.0,20.0,Dirty,X,2.5,S,Y,N,1,Others,980.00,2.0,2.8,H*,58,58,SC1D,82.0,75+,N,BankAccount,MonthlyAutoPay,11000,Direct Mail,Winback,Other than Non-Branded Search,P,1,1,21045.0,21045.0,21045.0,0
3,1044629296,FL,9/24/2024,3/24/2025,R,2079.0,8,NaN,NaN,12,Y,0.0,500.0,Clean,X,1.0,M,N,N,2,Geico,1039.50,5.0,6.2,H*,64,71,MC C=D,27.0,14-60,N,BankAccount,MonthlyAutoPay,10000,NaN,NaN,Other than Non-Branded Search,P,1,0,25095.0,29705.0,27400.0,0
4,1053922228,SC,7/13/2024,1/13/2025,R,3925.0,1,NaN,PL24,1,Y,0.0,50.0,Dirty,X,0.5,M,Y,Y,3,Others,1308.33,5.0,3.9,0,39,61,MCMCTD,15.0,14-60,N,BankAccount,MonthlyAutoPay,12453,Search,Non Branded,Search Non Branded,I,1,1,22970.0,32190.0,27392.0,0


---

## Stage 1 — Problem Framing & Target Definition  *(5 pts)*

**Produce in writing (no code required):**
- One-paragraph problem statement: what would a model predict, for whom, and why?
- Explicit target definition: variable name, type (binary / multiclass / continuous / count / ordinal), unit of observation, **time-of-prediction** (what is known *at prediction time* vs. observed afterward).
- Proposed primary evaluation metric **with justification** (why F1 vs. ROC-AUC vs. PR-AUC; why RMSE vs. MAE vs. MAPE) based on what you know of the problem at this early stage.
- Statement of single- vs. multi-target / multi-output problem based on what you know of the problem at this early stage.

### Your Stage 1 response

*Write your problem framing here.*

**Problem statement:**

**Target definition:**

**Time-of-prediction:**

**Proposed metric and justification:**

**Single- vs. multi-target:**

---

## Stage 2 — Data Inventory & Schema Audit  *(7 pts)*

**Produce:**
- Shape, memory footprint, dtype table.
- A **semantic feature-type table** (one row per column): `name | pandas dtype | semantic type | role | notes`, where  
  `semantic type ∈ {continuous, count, ordinal, nominal, binary, datetime, text, identifier, geographic, other}`  
  `role ∈ {feature, target, identifier, metadata, leakage-suspect}`.
- A list of columns where pandas dtype disagrees with the semantic type (e.g., zip codes as `int64`) and your planned coercion.

This table seeds your final Data Card.

In [ ]:
# Shape, memory, dtypes


In [ ]:
# Semantic feature-type table (one row per column)


In [ ]:
# Dtype-vs-semantic disagreements + coercion plan


### Stage 2 — Interpretation

*Summarize what the schema tells you about this dataset, and list the coercions you will apply downstream.*

---

## Stage 3 — Data Quality Assessment  *(8 pts)*

**Produce:**
- Exact and approximate (near-)duplicate-row counts; justify your approximate-duplicate key.
- Constant / quasi-constant columns (variance ≈ 0 or single value > 95%).
- ID-like columns (cardinality ≈ row count).
- Impossible / out-of-domain values (negatives where impossible, future dates, percentages > 100, etc.).
- Encoding glitches: whitespace, mixed case, mojibake, inconsistent units.
- For each issue: severity (low / medium / high) and remediation (drop / cap / impute / flag).

In [ ]:
# Duplicates (exact + approximate), constant cols, ID-like cols


In [ ]:
# Impossible / out-of-domain values


In [ ]:
# Encoding glitches (whitespace, case, mojibake, units)


### Stage 3 — Issue Register

| Issue | Affected column(s) | Severity | Remediation |
|---|---|---|---|
|  |  |  |  |

---

## Stage 4 — Missingness Analysis  *(8 pts)*

**Produce:**
- Per-column missing count and percentage; row-completeness distribution.
- Missingness matrix and missingness-correlation heatmap (e.g., `missingno`).
- **Missingness-vs-target test** for each column with non-trivial missingness:
  - Classification target: chi-square (with Cramér's V).
  - Regression target: Mann–Whitney or Kruskal–Wallis (with effect size, e.g., η² or rank-biserial).
- A reasoned MCAR / MAR / MNAR posture for the most-missing columns (argue, don't assert).
- Per-column imputation plan with justification.

In [ ]:
# Missingness counts, percentages, row-completeness distribution


In [ ]:
# Missingness matrix / correlation heatmap


In [ ]:
# Missingness-vs-target tests with effect sizes


### Stage 4 — Imputation Plan

| Column | % missing | Mechanism (MCAR/MAR/MNAR) + reasoning | Imputation strategy | Justification |
|---|---|---|---|---|
|  |  |  |  |  |

---

## Stage 5 — Univariate Analysis  *(8 pts)*

**Numeric features:**
- Summary statistics including count, mean, median, std, min, max, IQR, **skewness**, **kurtosis**.
- Distribution plots (histogram + KDE, or ECDF) for at least 8 numeric features.
- For features with |skew| > 1 or heavy tails: propose a transformation (log, log1p, Box-Cox, Yeo-Johnson) and **show the post-transformation distribution**.

**Categorical features:**
- Cardinality table.
- Frequency tables for low-cardinality categoricals.
- **Rare-level audit** for high-cardinality categoricals: define your "rare" threshold and propose grouping into "Other".
- Identify any categoricals masquerading as numeric (e.g., zip codes).

In [ ]:
# Numeric: summary stats including skew & kurtosis


In [ ]:
# Numeric: distribution plots (8+ features)


In [ ]:
# Numeric: transformation candidates (before/after)


In [ ]:
# Categorical: cardinality, frequency tables, rare-level audit


### Stage 5 — Interpretation

*Which features need transformation? Which categoricals need rare-level grouping? Any "numeric-looking" categoricals you reclassified?*

---

## Stage 6 — Target Analysis  *(10 pts)*

**Classification target:**
- Class frequencies and proportions; bar chart.
- Imbalance ratio.
- Recommended response (none / class weights / resampling / threshold tuning), **tied to the metric chosen in Stage 1**.
- Stratification implications for your Stage 10 split.

**Regression target:**
- Distribution, summary statistics, skew, kurtosis.
- Outlier and zero-inflation check.
- Transformation evaluation (e.g., log-target) with before/after plots; recommend whether to model on the transformed scale.

**Both:**
- If a temporal axis exists, plot target over time and comment on drift, seasonality, or regime change.

In [ ]:
# Target distribution


In [ ]:
# Imbalance / skew metrics + (if applicable) target-over-time plot


### Stage 6 — Interpretation

*State your recommended response (class weights / resampling / target transform / etc.) and tie it back to the metric chosen in Stage 1.*

---

## Stage 7 — Bivariate Signals vs. Target  *(12 pts)*

For **every retained feature**, quantify its bivariate relationship with the target using a test appropriate to the variable-type pair.

| Feature type | Classification target | Regression target |
|---|---|---|
| Continuous / count | Point-biserial *r*; ANOVA F or Kruskal-Wallis H; AUC of feature alone | Pearson *r*, Spearman *ρ* (report both) |
| Ordinal | Mann-Whitney / Kruskal-Wallis; Spearman *ρ* | Spearman *ρ*, Kendall *τ* |
| Nominal | Chi-square + **Cramér's V**; target rate by level | One-way ANOVA + **η²**; group means with CIs |
| Binary | Two-proportion z; difference in target rate | t-test or Mann-Whitney; **Cohen's *d*** |

**Required output:**
- A ranked table of all features with: test used, statistic, p-value, **effect size**, one-line interpretation.
- At least 6 illustrative plots (violin/box-by-class, target-rate-by-bin, scatter with LOWESS).
- Explicit acknowledgement of multiple-comparison risk (Bonferroni, Benjamini-Hochberg, or a stated screening caveat).

> A statistically significant *p*-value with a trivial effect size is **not a signal**. Effect size is required.

In [ ]:
# Bivariate tests — numeric features vs target


In [ ]:
# Bivariate tests — categorical features vs target (Cramér's V / η²)


In [ ]:
# Ranked feature table: feature | test | statistic | p | effect size | interpretation


In [ ]:
# Illustrative plots (6+) with takeaway captions


### Stage 7 — Interpretation

*Which features show meaningful effect sizes (not just significance)? Name at least one significant-but-trivial feature and one borderline-significant-but-practically-interesting feature. Address multiple-comparison risk.*

---

## Stage 8 — Multivariate Structure  *(12 pts)*

**Produce:**
- **Pearson and Spearman** correlation matrices for numeric features (heatmaps). Interpret disagreements (linear vs. monotonic).
- Multicollinearity assessment via **VIF** (or condition number) on the candidate numeric feature set; flag VIF > 10.
- Pairplots or facetted scatter for the top 6–8 features identified in Stage 7, colored/grouped by target.
- **Multivariate outlier detection**: Isolation Forest *or* Mahalanobis distance on the numeric feature set. Investigate the top 1% — errors, edge cases, or a meaningful subpopulation?
- *Encouraged but optional:* PCA scree plot and 2-component projection colored by target.

In [ ]:
# Pearson + Spearman correlation matrices; comment on disagreements


In [ ]:
# VIF / multicollinearity assessment


In [ ]:
# Pairplots / facets on top features by target


In [ ]:
# Multivariate outlier detection (Isolation Forest or Mahalanobis) + top-1% investigation


### Stage 8 — Interpretation

*Where do Pearson and Spearman disagree, and what does that imply? Which VIF clusters need consolidation? What did the multivariate outliers turn out to be?*

---

## Stage 9 — Mutual Information & Leakage Screen  *(12 pts)*

### 9a. Mutual Information
- Compute MI between every feature and the target using the **correct** sklearn estimator (`mutual_info_classif` for classification, `mutual_info_regression` for regression).
- Encode categoricals appropriately and pass `discrete_features` correctly.
- Rank features by MI; produce a horizontal bar chart of the top 20.
- Compare the MI ranking with Stage 7's linear-association ranking. **Identify and discuss ≥ 3 features where the two rankings disagree** — this is where non-linear signal lives.

### 9b. Leakage Screen (graded heavily)
For every feature in the top quartile of MI *or* top quartile of bivariate effect size, answer **in writing** for each feature:
1. Is this feature available **at the time of prediction**, or only after the target is realized?
2. Is it derived from, or a near-restatement of, the target?
3. Is it an identifier, timestamp, or post-event flag?
4. Does it look "too good to be true" given domain context?

Any feature failing 1–4 must be flagged **leakage-suspect** in your Data Card and excluded from the modeling set.

### 9c. Tree-based Sanity Check (Optional)
Train a single tree-based "EDA model" (e.g., `RandomForestClassifier`/`RandomForestRegressor`) **purely as an importance probe**. Report **permutation importance** on a held-out split. State the caveat: this is *not* model selection.

In [ ]:
# 9a — Mutual information (correct estimator, encoded categoricals, discrete_features mask)


In [ ]:
# 9a — Top-20 MI bar chart + comparison vs Stage 7 linear ranking


### 9b — Leakage Screen (per-feature)

*For every top-quartile feature, answer the four leakage questions and decide: keep / flag / drop.*

| Feature | Q1 (available at prediction time?) | Q2 (target-derived?) | Q3 (ID/timestamp/post-event?) | Q4 (too good to be true?) | Decision |
|---|---|---|---|---|---|
|  |  |  |  |  |  |

In [ ]:
# 9c — Tree-based sanity check + permutation importance on held-out split
# REMINDER: this is a non-linear sanity check on the MI ranking, NOT model selection.


### Stage 9 — Interpretation

*Where do MI and linear correlation disagree, and what is the non-linear story? Which features did you flag as leakage-suspect, and on what evidence? Does the permutation-importance ranking support or challenge your MI ranking?*

---

## Stage 10 — SL-Readiness Deliverables  *(15 pts)*

Produce **all four** sub-deliverables. The Data Card (10a) should also be exported as a standalone `<lastname>_<firstname>_data_card.md`.

### 10a — Data Card / Feature Dictionary

One row per feature. Internally consistent with Stages 2–9.

| Feature | Semantic type | % missing | Univariate notes | Bivariate signal (stat, effect size) | MI | Multicoll. flag | Leakage flag | Recommended treatment | Encoding | Scaling | Keep / Drop |
|---|---|---|---|---|---|---|---|---|---|---|---|
|  |  |  |  |  |  |  |  |  |  |  |  |

### 10b — Train / Validation / Test Split Recommendation

Choose **one** and justify against this dataset's specific risks:
- ☐ Random
- ☐ Stratified (by target)
- ☐ Grouped (specify grouping key: ____)
- ☐ Time-based (specify cutoff: ____)
- ☐ Nested

**Fractions:** train ___ / val ___ / test ___

**Justification (address leakage between splits, target stratification, group leakage, temporal validity):**

### 10c — Preprocessing Pipeline Sketch

Map each kept feature to its transformer chain. Specific enough that someone else could implement it as a `sklearn.pipeline.Pipeline` / `ColumnTransformer`. Mark fit-on-train-only operations clearly.

| Feature | Imputation | Transform | Encoding | Scaling | Notes (fit-on-train-only?) |
|---|---|---|---|---|---|
|  |  |  |  |  |  |

### 10d — Top-10 Candidate Predictors + Known Risks

**Top 10 (with evidence cited from Stages 7 + 9 + 8):**
1. 
2. 
3. 
4. 
5. 
6. 
7. 
8. 
9. 
10. 

**Known Risks register:**
- *Leakage suspects retained or removed:*
- *Multicollinearity clusters:*
- *Missingness assumptions:*
- *Drift / temporal-stability concerns:*
- *Fairness-sensitive features and proxies:*
- *Other:*

---

## Submission Checklist

- [ ] Notebook executes top-to-bottom on a fresh kernel.
- [ ] Random seed set and documented.
- [ ] All 10 stages have numbered headers.
- [ ] Data Card exported as both notebook table and standalone `.md`.
- [ ] Readiness Memo PDF (3–5 pages) addresses: problem framing, dataset health, top signals, leakage / risk register, split & preprocessing recommendation, open questions.
- [ ] Every p-value paired with an effect size.
- [ ] Every plot has title, axis labels, and a caption.
- [ ] Leakage screen explicit and per-feature for every top-quartile feature.
- [ ] Train / val / test split strategy stated and justified.
- [ ] Preprocessing pipeline sketch present.
- [ ] Files named per convention; submitted as a single zipped folder.